# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd


hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [hf_token])

print("Warehouse connected.")

Warehouse connected.


In [3]:
# Quick sanity check: does the table exist and have the right shape?
check = con.sql("""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as first_date,
    MAX(report_date) as last_date,
    COUNT(DISTINCT content_hash_id) as unique_content,
    COUNT(DISTINCT client_hash_id) as unique_clients
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

print(check)
# Expected: ~9.8M rows in March, ~520k unique content, ~104 clients

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows first_date  last_date  unique_content  unique_clients
0     9841378 2026-03-01 2026-03-31          331437              55


In [4]:
# Aggregating from daily grain to one row per content per month
con.sql("""
CREATE OR REPLACE TABLE content_march AS
SELECT
    content_hash_id,
    client_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS ctr,
    AVG(gsc_avg_position) AS avg_position,
    SUM(ga4_pageviews) AS pageviews,
    SUM(ga4_total_engagement_sec) AS engagement_sec,
    SUM(scroll_events) AS scroll_events,
    COUNT(*) FILTER (WHERE gsc_data_available) AS gsc_days,
    COUNT(*) AS total_days
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY content_hash_id, client_hash_id
HAVING SUM(gsc_impressions) IS NOT NULL
""")

result = con.sql("SELECT COUNT(*) as content_count FROM content_march").df()
print(f"Aggregated table created: {result['content_count'][0]} content items with GSC data")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aggregated table created: 331437 content items with GSC data


In [5]:
# Comapring fixed bins vs quantile bins

sig_a = con.sql("""
SELECT
    CASE WHEN avg_position <= 3  THEN '1_top3'
         WHEN avg_position <= 10 THEN '2_top10'
         WHEN avg_position <= 20 THEN '3_top20'
         ELSE '4_beyond20' END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(ctr), 4) AS avg_ctr,
    ROUND(MIN(avg_position), 1) AS min_pos,
    ROUND(MAX(avg_position), 1) AS max_pos
FROM content_march
WHERE impressions >= 10
GROUP BY 1
ORDER BY 1
""").df()

print(sig_a)
print(f"\nBucket size ratio (max/min): {sig_a['n'].max() / sig_a['n'].min():.1f}x")

  position_bucket      n  avg_ctr  min_pos  max_pos
0          1_top3  11154   0.0042      0.0      3.0
1         2_top10  64211   0.0038      3.0     10.0
2         3_top20  28967   0.0025     10.0     20.0
3      4_beyond20  38874   0.0014     20.0    115.7

Bucket size ratio (max/min): 5.8x


In [6]:
# Computing quantile bins for position

quantile_query = con.sql("""
WITH position_quartiles AS (
  SELECT
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY avg_position) AS q1_cutoff,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY avg_position) AS q2_cutoff,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY avg_position) AS q3_cutoff
  FROM content_march
  WHERE impressions >= 10
)
SELECT
  CASE
    WHEN avg_position <= q1_cutoff THEN '1_top_25pct'
    WHEN avg_position <= q2_cutoff THEN '2_top50pct'
    WHEN avg_position <= q3_cutoff THEN '3_top75pct'
    ELSE '4_bottom_25pct'
  END AS position_bucket,
  COUNT(*) AS n,
  ROUND(AVG(ctr), 4) AS avg_ctr,
  ROUND(MIN(avg_position), 1) AS min_pos,
  ROUND(MAX(avg_position), 1) AS max_pos,
  q1_cutoff, q2_cutoff, q3_cutoff
FROM content_march, position_quartiles
WHERE impressions >= 10
GROUP BY 1, q1_cutoff, q2_cutoff, q3_cutoff
ORDER BY 1
""").df()

print(quantile_query)
print("\n--- Cutoff values ---")
print(f"Q1 cutoff (25th percentile): {quantile_query['q1_cutoff'].iloc[0]:.2f}")
print(f"Q2 cutoff (50th percentile): {quantile_query['q2_cutoff'].iloc[0]:.2f}")
print(f"Q3 cutoff (75th percentile): {quantile_query['q3_cutoff'].iloc[0]:.2f}")

  position_bucket      n  avg_ctr  min_pos  max_pos  q1_cutoff  q2_cutoff  \
0     1_top_25pct  35802   0.0044      0.0      5.3   5.337441   9.281279   
1      2_top50pct  35801   0.0034      5.3      9.3   5.337441   9.281279   
2      3_top75pct  35801   0.0025      9.3     21.6   5.337441   9.281279   
3  4_bottom_25pct  35802   0.0013     21.6    115.7   5.337441   9.281279   

   q3_cutoff  
0  21.608828  
1  21.608828  
2  21.608828  
3  21.608828  

--- Cutoff values ---
Q1 cutoff (25th percentile): 5.34
Q2 cutoff (50th percentile): 9.28
Q3 cutoff (75th percentile): 21.61


In [7]:
# Signal Check B - impression volume distribution

sig_b = con.sql("""
SELECT
    CASE WHEN impressions < 100   THEN '1_low'
         WHEN impressions < 1000  THEN '2_mid'
         WHEN impressions < 10000 THEN '3_high'
         ELSE '4_veryhigh' END AS impression_bucket,
    COUNT(*) AS n,
    ROUND(AVG(avg_position), 1) AS avg_position,
    ROUND(AVG(ctr), 4) AS avg_ctr
FROM content_march
GROUP BY 1 ORDER BY 1
""").df()

print(sig_b)
print(f"\nBucket size ratio (max/min): {sig_b['n'].max() / sig_b['n'].min():.1f}x")

  impression_bucket       n  avg_position  avg_ctr
0             1_low  229996          18.1   0.0073
1             2_mid   56383          17.2   0.0023
2            3_high   39181          10.9   0.0030
3        4_veryhigh    5877          11.6   0.0029

Bucket size ratio (max/min): 39.1x


In [11]:
# Cell 8: Top-20 review

# Get the top 20 highest-scored items
top_20 = content_full[content_full['action_label'] == 'FIX_TITLE_META_HIGH_PRIORITY'].head(20)

print("TOP 20 HIGHEST-SCORED PICKS")
print("=" * 120)
print()

for idx, (i, row) in enumerate(top_20.iterrows(), 1):
    print(f"{idx}. content_hash_id: {row['content_hash_id']}")
    print(f"   Action: {row['action_label']}")
    print(f"   Reason: {row['reason_code']}")
    print(f"   Score: {row['score']:.1f}")
    print(f"   ---")
    print(f"   Impressions: {row['impressions']:.0f}")
    print(f"   Clicks: {row['clicks']:.0f}")
    print(f"   Actual CTR: {row['ctr']:.4f} ({row['ctr']*100:.2f}%)")
    print(f"   Expected CTR for position {row['avg_position']:.1f}: {row['expected_ctr']:.4f} ({row['expected_ctr']*100:.2f}%)")
    print(f"   CTR gap: {row['ctr_gap']:.4f} ({row['ctr_gap']*100:.2f}%)")
    print(f"   Position bucket: {row['position_bucket']}")
    print(f"   Avg position: {row['avg_position']:.1f}")
    print(f"   Days active: {row['days_active']:.0f} days")
    print()

# Summary stats
print("=" * 120)
print(f"Average impressions in top 20: {top_20['impressions'].mean():.0f}")
print(f"Average position in top 20: {top_20['avg_position'].mean():.1f}")
print(f"Average CTR gap in top 20: {top_20['ctr_gap'].mean():.4f}")

TOP 20 HIGHEST-SCORED PICKS

1. content_hash_id: content_acbcc847f8996314
   Action: FIX_TITLE_META_HIGH_PRIORITY
   Reason: CTR_BELOW_EXPECTED_FOR_POSITION
   Score: 4176.1
   ---
   Impressions: 1055962
   Clicks: 1424
   Actual CTR: 0.0013 (0.13%)
   Expected CTR for position 4.0: 0.0053 (0.53%)
   CTR gap: 0.0040 (0.40%)
   Position bucket: 1_top_25pct
   Avg position: 4.0
   Days active: 195 days

2. content_hash_id: content_e241d6415ac9e534
   Action: FIX_TITLE_META_HIGH_PRIORITY
   Reason: CTR_BELOW_EXPECTED_FOR_POSITION
   Score: 3986.8
   ---
   Impressions: 1829204
   Clicks: 5714
   Actual CTR: 0.0031 (0.31%)
   Expected CTR for position 4.9: 0.0053 (0.53%)
   CTR gap: 0.0022 (0.22%)
   Position bucket: 1_top_25pct
   Avg position: 4.9
   Days active: 499 days

3. content_hash_id: content_21309e9a83c83653
   Action: FIX_TITLE_META_HIGH_PRIORITY
   Reason: CTR_BELOW_EXPECTED_FOR_POSITION
   Score: 3424.7
   ---
   Impressions: 983861
   Clicks: 1793
   Actual CTR: 0.0018 (0.1

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal check A: CTR vs position

This rule checks if a better ranked page gets more clicks.

```
  position_bucket      n  avg_ctr  min_pos  max_pos  q1_cutoff q2_cutoff
0     1_top_25pct  35802   0.0044      0.0      5.3   5.337441   9.281279   
1      2_top50pct  35801   0.0034      5.3      9.3   5.337441   9.281279   
2      3_top75pct  35801   0.0025      9.3     21.6   5.337441   9.281279   
3  4_bottom_25pct  35802   0.0013     21.6    115.7   5.337441   9.281279
```

 **Verdict: CONFIRMED**

 CTR drops monotonically across all quartiles (0.44% → 0.34% → 0.25% → 0.13%). The pattern is clean and consistent: pages that rank worse genuinely get proportionally fewer clicks. Each bucket has approximately equal rows, so the averages are computed on balanced data, so the result is trustworthy.

## Signal check B: Impression volume

This rule checks if there are pages that are getting traction but don't rank well.

```
  impression_bucket       n  avg_position  avg_ctr
0             1_low  229996          18.1   0.0073
1             2_mid   56383          17.2   0.0023
2            3_high   39181          10.9   0.0030
3        4_veryhigh    5877          11.6   0.0029
```

**Verdict: MIXED**

High and very-high impression pages are sitting at position 10.9–11.6, meaning there's real headroom for improvement as they're not all already ranked at top-3. That confirms an opportunity exists for high-traffic content. However, 69% of the corpus has  (<100) impressions, which are too small to diagnose as their CTR ratios are too volatile for actionable signals. The signal works for the subset that matters (the high-volume pages), but most content is too young or niche to benefit from title fixes. Volume validates the existence of a middle-tier pool, but doesn't tell the whole story.

## Reason Code:

**CTR_BELOW_EXPECTED_FOR_POSITION** — Every flagged item underperforms the CTR baseline for its position bucket.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# Rule Encoding:

**score = (expected_ctr - actual_ctr) × impressions**

Where:
*   **expected_ctr** is the average CTR for pages in the same position quartile (computed from full warehouse, all months)

*   **actual_ctr** is the page's observed CTR (clicks ÷ impressions)
The gap is clipped at zero (pages matching or exceeding expected CTR score 0)

*   **impressions** acts as a multiplier — bigger audience makes even small gaps high-value

In [10]:
# Build the baseline scored table (full warehouse, all months)

# Step 1: Compute expected CTR per position quantile
expected_ctr_table = con.sql('''
SELECT
  CASE
    WHEN gsc_avg_position <= 5.337441 THEN '1_top_25pct'
    WHEN gsc_avg_position <= 9.281279 THEN '2_top50pct'
    WHEN gsc_avg_position <= 21.608828 THEN '3_top75pct'
    ELSE '4_bottom_25pct'
  END AS position_bucket,
  AVG(gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0)) AS expected_ctr
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE gsc_impressions > 0
GROUP BY 1
''').df()

print("Expected CTR per position bucket:")
print(expected_ctr_table)

# Step 2: Aggregate to one row per content (full warehouse)
content_full = con.sql('''
SELECT
  content_hash_id,
  client_hash_id,
  SUM(gsc_impressions) AS impressions,
  SUM(gsc_clicks) AS clicks,
  SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS ctr,
  AVG(gsc_avg_position) AS avg_position,
  COUNT(DISTINCT report_date) AS days_active,
  MIN(report_date) AS first_date,
  MAX(report_date) AS last_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE gsc_impressions > 0
GROUP BY content_hash_id, client_hash_id
HAVING SUM(gsc_impressions) >= 100
''').df()

print(f"\nContent items aggregated: {len(content_full)}")

# Step 3: Merge and compute score
content_full['position_bucket'] = content_full['avg_position'].apply(
  lambda x: '1_top_25pct' if x <= 5.337441 else
            '2_top50pct' if x <= 9.281279 else
            '3_top75pct' if x <= 21.608828 else
            '4_bottom_25pct'
)

content_full = content_full.merge(expected_ctr_table, on='position_bucket', how='left')

content_full['ctr_gap'] = (content_full['expected_ctr'] - content_full['ctr']).clip(lower=0)
content_full['score'] = content_full['ctr_gap'] * content_full['impressions']
content_full['reason_code'] = 'CTR_BELOW_EXPECTED_FOR_POSITION'

# Temporary action labels (we'll adjust thresholds after seeing distribution)
content_full['action_label'] = content_full['score'].apply(
  lambda s: 'FIX_TITLE_META_HIGH_PRIORITY' if s >= 100 else
            'FIX_TITLE_META_MONITOR' if s >= 20 else
            'NO_ACTION_CTR_ON_TRACK'
)

# Sort by score
content_full = content_full.sort_values('score', ascending=False)

print(f"\nScore distribution:")
print(content_full['score'].describe())

print(f"\nAction label breakdown:")
print(content_full['action_label'].value_counts())

# Write to CSV
import os
os.makedirs('work/outputs', exist_ok=True)
content_full.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nCSV written to work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Expected CTR per position bucket:
  position_bucket  expected_ctr
0  4_bottom_25pct      0.001673
1      2_top50pct      0.003892
2     1_top_25pct      0.005303
3      3_top75pct      0.003393


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Content items aggregated: 182135

Score distribution:
count    182135.000000
mean         13.066198
std          59.851503
min           0.000000
25%           0.000000
50%           0.806188
75%           5.069511
max        4176.069381
Name: score, dtype: float64

Action label breakdown:
action_label
NO_ACTION_CTR_ON_TRACK          160982
FIX_TITLE_META_MONITOR           15882
FIX_TITLE_META_HIGH_PRIORITY      5271
Name: count, dtype: int64

CSV written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.